<a href="https://colab.research.google.com/github/Samikhxnn/customdataset_pytorch_classifier/blob/main/CustomDataset(classification)_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# import libraries and modules

In [ ]:
import torch
from torch import nn  # nn provides layers and loss functions
from torch.utils.data import Dataset     # for creating custom dataset class
from torch.utils.data import DataLoader




In [ ]:

import torchvision
from torchvision import transforms
from torchvision import models


In [ ]:
import os
from PIL import Image

# create a custom dataset class

In [ ]:
class Super(Dataset):
  def __init__(self,root,transform):
    self.root=root
    self.transform=transform



    # create class to index dictionary for giving numeric values to class names
    self.classes=sorted(os.listdir(root))
    self.class_to_idx={class_name:i  for i,class_name in enumerate(self.classes)}

    # create a list (images)  where each element is a pair of img path and label
    self.images=[]
    for class_name in self.classes:
       class_path=os.path.join(root,class_name)

       for img_name in os.listdir(class_path):
        img_path=os.path.join(class_path,img_name)

        self.images.append(      (img_path,self.class_to_idx[class_name])    )

  def __len__(self):
    return len(self.images)

  def __getitem__(self,idx):
    img_path,label=self.images[idx]

    img=Image.open(img_path).convert("RGB")

    if self.transform:
      img=self.transform(img)

    return img,label




# set up train and test transforms

In [ ]:
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomCrop(224,padding=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
test_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# create instance of the class

In [ ]:
# create instance of the class
train_dataset=Super(root='/content/drive/MyDrive/super/train',transform=train_transform)
test_dataset=Super(root='/content/drive/MyDrive/super/test',transform=test_transform)

# prepare DataLoader

In [ ]:
train_dataloader=DataLoader(train_dataset,batch_size=10,shuffle=True)
test_dataloader=DataLoader(test_dataset,batch_size=5)

# loading a pretrained Model

In [ ]:
model=models.vgg16(pretrained=True)

In [ ]:
# freezing feature layer parameters
for param in model.features.parameters():
  param.requires_grad=False

# to sure classifier layer is learning
for param in model.classifier.parameters():
  param.requires_grad=True

In [ ]:
model.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [ ]:
# updating classifier layer according to our data
model.classifier=nn.Sequential(
    nn.Linear(25088,1000),
    nn.ReLU(),
    nn.Linear(1000,300),
    nn.ReLU(),
    nn.Linear(300,3)
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set up loss function and optimizer

In [ ]:
loss_func=nn.CrossEntropyLoss()
o=torch.optim.Adam(params=model.classifier.parameters(),
                   lr=0.0001
                   )

# training Loop

In [ ]:
epochs=11

for epoch in range(epochs):
  t_loss=0
  for x,y in train_dataloader:
    x, y = x.to(device), y.to(device)
    o.zero_grad()

    y_logits=model(x)
    loss=loss_func(y_logits,y)

    t_loss+=loss

    #o.zero_grad()
    loss.backward()
    o.step()

  if epoch%2==0:
    print("epoch",epoch)
    print("avg loss per batch : ",t_loss/len(train_dataloader))

epoch 0
avg loss per batch :  tensor(0.4779, grad_fn=<DivBackward0>)
epoch 2
avg loss per batch :  tensor(0.0596, grad_fn=<DivBackward0>)
epoch 4
avg loss per batch :  tensor(0.0546, grad_fn=<DivBackward0>)
epoch 6
avg loss per batch :  tensor(0.0146, grad_fn=<DivBackward0>)
epoch 8
avg loss per batch :  tensor(0.0399, grad_fn=<DivBackward0>)
epoch 10
avg loss per batch :  tensor(0.0171, grad_fn=<DivBackward0>)


# testing loop

In [ ]:
model.eval()

with torch.no_grad():



  total_correct=0
  total_sample=0
  for x,y in test_dataloader:
    x, y = x.to(device), y.to(device)
    y_logits=model(x)
    y_prob=torch.softmax(y_logits,dim=1)
    y_class=torch.argmax(y_prob,dim=1)

    total_correct+=(y==y_class).sum().item()
    total_sample+=y.size(0)


  print("accuracy",total_correct*100/total_sample)


accuracy 87.77777777777777
